# Pilot C — 02: oracle ablation pilot (GPU, L4 x4)
e4b on the 15 pilot tasks: O0, O-func, O-all, then O0 again (noise estimate). Optional e2b block (set RUN_E2B).
Hints are appended to the problem statement in a local task copy only (never in a submission).

Accelerator: **GPU L4 x4**. Inputs: competition, wheelhouse dataset, Gemma 4 e4b (and e2b) QAT models.

In [ ]:
import os
os.makedirs('/kaggle/working/pilot/common', exist_ok=True)
os.makedirs('/kaggle/working/pilot/pilot_c', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/pilot/common/__init__.py

In [ ]:
%%writefile /kaggle/working/pilot/common/diffs.py
"""Unified-diff parsing helpers (no task data, no pilot-specific logic).

Parses the `--- a/X / +++ b/X / @@ ... @@` diffs used in the competition's
`patch` and `test_patch` fields into per-file change records with old-side
line numbers, so callers can map changes onto the base snapshot.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

_HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")

# Mirrors the scorer's notion of protected test files (HARNESS_README §8.1).
_TEST_DIR_PARTS = {"tests", "test", "testing"}


def is_test_path(path: str) -> bool:
    parts = path.split("/")
    name = parts[-1]
    if name.startswith("test_") and name.endswith(".py"):
        return True
    if name.endswith("_test.py"):
        return True
    return name.endswith(".py") and any(p in _TEST_DIR_PARTS for p in parts[:-1])


@dataclass
class Insertion:
    """Lines added after old line `after_old` (0 = at file start)."""

    after_old: int
    lines: list[str]


@dataclass
class FileDiff:
    old_path: str | None
    new_path: str | None
    removed_old_lines: list[int] = field(default_factory=list)  # old-side line numbers deleted/replaced
    insertions: list[Insertion] = field(default_factory=list)
    hunks: int = 0
    added: int = 0
    removed: int = 0
    all_hunks_from_zero: bool = True  # every hunk is "@@ -0,0 ..." (file absent on old side)

    @property
    def path(self) -> str:
        return self.new_path or self.old_path or ""

    @property
    def maybe_new_file(self) -> bool:
        return self.old_path is None or (self.hunks > 0 and self.all_hunks_from_zero)

    @property
    def deleted_file(self) -> bool:
        return self.new_path is None


def _strip_prefix(p: str) -> str | None:
    p = p.split("\t", 1)[0].strip()
    if p == "/dev/null":
        return None
    if p.startswith('"') and p.endswith('"'):
        p = p[1:-1]
    if p.startswith(("a/", "b/")):
        p = p[2:]
    return p


def parse_unified_diff(text: str) -> list[FileDiff]:
    """Parse a unified diff into FileDiff records (robust to missing `diff --git` headers)."""
    files: list[FileDiff] = []
    cur: FileDiff | None = None
    lines = text.splitlines()
    i = 0
    old_ln = new_ln = 0
    rem_old = rem_new = 0
    pending: Insertion | None = None

    def flush_pending() -> None:
        nonlocal pending
        if pending is not None and cur is not None:
            cur.insertions.append(pending)
        pending = None

    while i < len(lines):
        line = lines[i]
        in_hunk = rem_old > 0 or rem_new > 0
        if not in_hunk and line.startswith("--- ") and i + 1 < len(lines) and lines[i + 1].startswith("+++ "):
            flush_pending()
            cur = FileDiff(old_path=_strip_prefix(line[4:]), new_path=_strip_prefix(lines[i + 1][4:]))
            files.append(cur)
            i += 2
            continue
        if not in_hunk and line.startswith("@@ ") and cur is not None:
            flush_pending()
            m = _HUNK_RE.match(line)
            if m:
                o_start = int(m.group(1))
                o_cnt = int(m.group(2)) if m.group(2) is not None else 1
                n_cnt = int(m.group(4)) if m.group(4) is not None else 1
                cur.hunks += 1
                if not (o_start == 0 and o_cnt == 0):
                    cur.all_hunks_from_zero = False
                # old_ln = number of the last old line consumed so far
                old_ln = o_start - 1 if o_cnt > 0 else o_start
                rem_old, rem_new = o_cnt, n_cnt
            i += 1
            continue
        if in_hunk and cur is not None:
            if line.startswith("\\"):  # "\ No newline at end of file"
                i += 1
                continue
            tag = line[:1]
            body = line[1:]
            if tag == "+":
                if pending is None or pending.after_old != old_ln:
                    flush_pending()
                    pending = Insertion(after_old=old_ln, lines=[])
                pending.lines.append(body)
                cur.added += 1
                rem_new -= 1
            elif tag == "-":
                flush_pending()
                old_ln += 1
                cur.removed_old_lines.append(old_ln)
                cur.removed += 1
                rem_old -= 1
            else:  # context (' ' or empty line inside hunk)
                flush_pending()
                old_ln += 1
                rem_old -= 1
                rem_new -= 1
            i += 1
            continue
        i += 1
    flush_pending()
    return files


_ADDED_TEST_DEF = re.compile(r"^\+\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)
_REMOVED_TEST_DEF = re.compile(r"^-\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)
_HUNK_HEADER_TEST = re.compile(r"^@@[^@]+@@\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\b", re.M)
_CONTEXT_TEST_DEF = re.compile(r"^ \s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)


def required_test_names(test_patch: str) -> list[str]:
    """Test function names the scorer requires to pass, derived from test_patch.

    Re-implements the documented rule of the scorer (swegemma 0.2.11,
    `_extract_test_functions_from_patch`): added `def test_*` lines if any;
    otherwise test functions named in hunk headers / context, minus removed ones.
    Verified equal to the scorer's function on all 129 dev tasks (see
    pilot_c/check_test_names.py).
    """
    if not test_patch:
        return []
    added = _ADDED_TEST_DEF.findall(test_patch)
    if added:
        return list(dict.fromkeys(added))
    removed = set(_REMOVED_TEST_DEF.findall(test_patch))
    headers = _HUNK_HEADER_TEST.findall(test_patch)
    context = _CONTEXT_TEST_DEF.findall(test_patch)
    ctx = set(context)
    out = []
    for fn in dict.fromkeys(headers + context):
        if fn in removed:
            continue
        if fn not in ctx and (any(r.startswith(fn) for r in removed) or any(c != fn and c.startswith(fn) for c in ctx)):
            continue
        out.append(fn)
    return out

In [ ]:
%%writefile /kaggle/working/pilot/common/locate.py
"""Map diff changes onto enclosing Python scopes (def/class) of the base file.

Given the base-commit source of a file and a parsed FileDiff, returns the
innermost enclosing function/class for every changed region, using `ast` on
the base file. Deleted/replaced lines map by line number; inserted blocks map
by their anchor line plus indentation (an insertion indented less than a
scope's body belongs to the parent scope).
"""
from __future__ import annotations

import ast
from dataclasses import dataclass

from .diffs import FileDiff


@dataclass(frozen=True)
class Scope:
    qualname: str  # e.g. "APIRoute.__init__" (relative to module)
    kind: str  # "function" | "class"
    start: int  # first line incl. decorators
    def_line: int
    end: int
    body_indent: int
    depth: int
    header_indent: int = 0


def module_path(file_path: str) -> str:
    """'src/requests/models.py' -> 'requests.models'; 'pkg/__init__.py' -> 'pkg'."""
    p = file_path[:-3] if file_path.endswith(".py") else file_path
    parts = [x for x in p.split("/") if x]
    if parts and parts[0] in ("src", "lib"):
        parts = parts[1:]
    if parts and parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


def collect_scopes(source: str) -> list[Scope]:
    tree = ast.parse(source)
    scopes: list[Scope] = []

    def visit(node: ast.AST, prefix: str, depth: int) -> None:
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{child.name}" if prefix else child.name
                start = min([child.lineno] + [d.lineno for d in child.decorator_list])
                body_indent = child.body[0].col_offset if child.body else child.col_offset + 4
                kind = "class" if isinstance(child, ast.ClassDef) else "function"
                scopes.append(Scope(q, kind, start, child.lineno, child.end_lineno or child.lineno, body_indent, depth, child.col_offset))
                visit(child, q, depth + 1)
            else:
                visit(child, prefix, depth)

    visit(tree, "", 0)
    return scopes


def _innermost(scopes: list[Scope], line: int) -> list[Scope]:
    """Scopes containing `line`, outermost first."""
    return sorted((s for s in scopes if s.start <= line <= s.end), key=lambda s: s.depth)


def _indent(s: str) -> int:
    return len(s) - len(s.lstrip(" \t"))


def report_scope(chain: list[Scope]) -> Scope | None:
    """Collapse a containment chain to what a developer would name:
    the outermost function (methods count as functions), or the innermost class
    if the change is in a class body outside any method."""
    if not chain:
        return None
    for s in chain:
        if s.kind == "function":
            return s
    return chain[-1]


def changed_scopes(source: str, fd: FileDiff) -> tuple[list[Scope], bool]:
    """Return (scopes touched by the diff, touches_module_level_code)."""
    scopes = collect_scopes(source)
    src_lines = source.splitlines()
    hits: dict[str, Scope] = {}
    module_level = False

    def last_code_line_at_or_before(n: int) -> int:
        while n >= 1 and (n > len(src_lines) or not src_lines[n - 1].strip()):
            n -= 1
        return n

    for ln in fd.removed_old_lines:
        s = report_scope(_innermost(scopes, ln))
        if s is None:
            module_level = True
        else:
            hits.setdefault(s.qualname, s)

    removed = set(fd.removed_old_lines)
    for ins in fd.insertions:
        code = [x for x in ins.lines if x.strip()]
        if not code:
            continue
        ind = _indent(code[0])
        if ins.after_old in removed:
            # replacement: the new lines take the place of removed ones, so they may
            # sit at a scope's header level (e.g. an edited decorator or signature)
            chain = [s for s in _innermost(scopes, ins.after_old) if s.header_indent <= ind]
        else:
            anchor = last_code_line_at_or_before(ins.after_old)
            chain = [s for s in _innermost(scopes, anchor) if s.body_indent <= ind] if anchor >= 1 else []
        s = report_scope(chain)
        if s is None:
            module_level = True
        else:
            hits.setdefault(s.qualname, s)

    ordered = sorted(hits.values(), key=lambda s: s.def_line)
    return ordered, module_level

In [ ]:
%%writefile /kaggle/working/pilot/common/tasks.py
"""Task loading and gold-location extraction (reusable; no pilot-specific logic).

Nothing here writes task data anywhere: callers decide where outputs go.
Competition data (tasks.jsonl, snapshots) must stay out of git.
"""
from __future__ import annotations

import json
import tarfile
from dataclasses import dataclass, field, asdict
from pathlib import Path

from .diffs import FileDiff, is_test_path, parse_unified_diff, required_test_names
from .locate import changed_scopes, module_path


def load_tasks(path: str | Path) -> list[dict]:
    out, seen = [], set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                t = json.loads(line)
                if t["instance_id"] not in seen:
                    seen.add(t["instance_id"])
                    out.append(t)
    return out


def read_snapshot_members(tgz: str | Path, wanted: set[str]) -> dict[str, str]:
    """Return {path: text} for the wanted repo-relative paths, reading the archive in one pass.
    Never extracts to disk and never executes anything from the archive."""
    found: dict[str, str] = {}
    want = set(wanted)
    with tarfile.open(tgz, mode="r|gz") as tf:
        for m in tf:
            name = m.name[2:] if m.name.startswith("./") else m.name
            if name in want and m.isfile():
                fh = tf.extractfile(m)
                if fh is not None:
                    found[name] = fh.read().decode("utf-8", errors="replace")
                if len(found) == len(want):
                    break
    return found


@dataclass
class GoldFunction:
    file: str
    qualname: str  # relative to module, e.g. "APIRoute.__init__"
    dotted: str  # e.g. "fastapi.routing.APIRoute.__init__" (graph-node style)
    kind: str  # "function" | "class"
    def_line: int


@dataclass
class GoldLocations:
    instance_id: str
    repo: str
    src_files: list[str] = field(default_factory=list)  # non-test .py files changed by the gold patch
    new_src_files: list[str] = field(default_factory=list)  # subset absent from the base snapshot
    other_files: list[str] = field(default_factory=list)  # non-.py or test files in the gold patch
    functions: list[GoldFunction] = field(default_factory=list)
    module_level_files: list[str] = field(default_factory=list)  # files with changes outside any def/class
    test_files: list[str] = field(default_factory=list)
    new_test_files: list[str] = field(default_factory=list)  # test files absent from the base snapshot
    test_names: list[str] = field(default_factory=list)  # scorer-required test functions
    gold_added: int = 0
    gold_removed: int = 0
    parse_errors: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return asdict(self)


def diff_scopes(diff_text: str, snapshot_tgz: str | Path, texts: dict[str, str] | None = None) -> list[str]:
    """Dotted names of the def/class scopes (in the base snapshot) changed by any diff,
    e.g. an agent's patch. Module-level changes are reported as '<module>:<path>'."""
    fds = [fd for fd in parse_unified_diff(diff_text) if fd.path.endswith(".py") and fd.old_path]
    if texts is None:
        texts = read_snapshot_members(snapshot_tgz, {fd.path for fd in fds})
    out: list[str] = []
    for fd in fds:
        if fd.path not in texts:
            continue
        try:
            scopes, mod_level = changed_scopes(texts[fd.path], fd)
        except SyntaxError:
            continue
        mod = module_path(fd.path)
        out += [f"{mod}.{s.qualname}" if mod else s.qualname for s in scopes]
        if mod_level:
            out.append(f"<module>:{fd.path}")
    return out


def gold_locations(task: dict, snapshot_tgz: str | Path | None) -> GoldLocations:
    """Derive gold file(s), enclosing function(s) and required test names for one task.

    Uses only *locations* from the gold patch (paths, line numbers) and test
    names from test_patch; the base-file AST comes from the snapshot.
    If snapshot_tgz is None, function-level fields stay empty.
    """
    g = GoldLocations(task["instance_id"], task["repo"])
    fdiffs: list[FileDiff] = parse_unified_diff(task["patch"])
    tdiffs: list[FileDiff] = parse_unified_diff(task["test_patch"])
    py_src = [fd for fd in fdiffs if fd.path.endswith(".py") and not is_test_path(fd.path)]
    g.src_files = list(dict.fromkeys(fd.path for fd in py_src))
    g.other_files = [fd.path for fd in fdiffs if fd not in py_src]
    g.gold_added = sum(fd.added for fd in fdiffs)
    g.gold_removed = sum(fd.removed for fd in fdiffs)
    g.test_files = list(dict.fromkeys(fd.path for fd in tdiffs if fd.new_path))
    g.test_names = required_test_names(task["test_patch"])

    if snapshot_tgz is None:
        return g
    wanted = {fd.path for fd in py_src if fd.old_path} | set(g.test_files)
    texts = read_snapshot_members(snapshot_tgz, wanted)
    g.new_test_files = [p for p in g.test_files if p not in texts]
    for fd in py_src:
        if fd.path not in texts:
            g.new_src_files.append(fd.path)
            continue
        try:
            scopes, mod_level = changed_scopes(texts[fd.path], fd)
        except SyntaxError as e:  # pragma: no cover - base files should parse
            g.parse_errors.append(f"{fd.path}: {e}")
            continue
        mod = module_path(fd.path)
        for s in scopes:
            g.functions.append(GoldFunction(fd.path, s.qualname, f"{mod}.{s.qualname}" if mod else s.qualname, s.kind, s.def_line))
        if mod_level:
            g.module_level_files.append(fd.path)
    return g

In [ ]:
%%writefile /kaggle/working/pilot/common/trace.py
"""Parse swegemma/adk-eval-core session traces into a flat event list.

Supports the ATIF v1.7 JSON written by `SessionTrace.save()` (the harness
default, `results_dir/traces/trace_<id>.json`) and the legacy
`{"entries": [...]}` format. Mirrors `SessionTrace.from_dict` in
adk-eval-core 0.1.0 without importing it, so this works on any machine.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any


@dataclass
class Event:
    type: str  # tool_call | tool_response | thinking | text | final | continuation_nudge | compaction | ...
    author: str = ""
    elapsed: float = 0.0
    tool: str = ""
    args: dict[str, Any] = field(default_factory=dict)
    result: str = ""
    content: str = ""


@dataclass
class ToolCall:
    index: int  # order among tool calls in the task (0-based)
    tool: str
    args: dict[str, Any]
    author: str
    elapsed: float
    result: str | None = None  # raw response text (JSON string) if matched

    def result_json(self) -> dict[str, Any] | None:
        if self.result is None:
            return None
        try:
            v = json.loads(self.result)
        except (ValueError, TypeError):
            return None
        # ADK sometimes wraps tool output as {"result": "<json string>"}
        if isinstance(v, dict) and set(v) == {"result"} and isinstance(v["result"], str):
            try:
                inner = json.loads(v["result"])
                if isinstance(inner, dict):
                    return inner
            except ValueError:
                pass
        return v if isinstance(v, dict) else None


def _atif_events(data: dict[str, Any]) -> list[Event]:
    evs: list[Event] = []
    for step in data.get("steps", []):
        source = step.get("source", "agent")
        msg = step.get("message")
        extra = step.get("extra") or {}
        author = extra.get("author", source)
        etype = extra.get("event_type")
        el = float(extra.get("elapsed_s", 0.0) or 0.0)
        if msg is not None or (etype is not None and not step.get("tool_calls") and not step.get("observation") and etype != "usage"):
            if not etype:
                etype = {"system": "system_instruction", "user": "task_prompt"}.get(source, "text")
            evs.append(Event(etype, author, el, content=str(msg) if msg is not None else ""))
        for tc in step.get("tool_calls", []) or []:
            tx = tc.get("extra") or {}
            evs.append(Event("tool_call", tx.get("author", author), float(tx.get("elapsed_s", el) or 0.0),
                             tool=tc.get("function_name", ""), args=tc.get("arguments") or {}))
        obs = step.get("observation")
        if obs:
            ox = obs.get("extra") or {}
            content = obs.get("content", "")
            if not isinstance(content, str):
                content = json.dumps(content, default=str)
            evs.append(Event("tool_response", ox.get("author") or "system", float(ox.get("elapsed_s", el) or 0.0),
                             tool=ox.get("tool_name", ""), result=content))
    return evs


def _legacy_events(data: dict[str, Any]) -> list[Event]:
    return [
        Event(e.get("type") or "custom", e.get("author") or "", float(e.get("elapsed_s") or 0.0),
              tool=e.get("tool") or "", args=e.get("args") or {}, result=e.get("result") or "", content=e.get("content") or "")
        for e in data.get("entries", [])
    ]


def parse_trace(data: dict[str, Any]) -> list[Event]:
    if str(data.get("schema_version", "")).startswith("ATIF"):
        return _atif_events(data)
    return _legacy_events(data)


def load_trace(path: str | Path) -> list[Event]:
    with open(path, encoding="utf-8") as f:
        return parse_trace(json.load(f))


def pair_tool_calls(events: list[Event]) -> list[ToolCall]:
    """Match each tool_response to the earliest unanswered tool_call with the same tool name."""
    calls: list[ToolCall] = []
    pending: dict[str, list[ToolCall]] = {}
    for ev in events:
        if ev.type == "tool_call":
            c = ToolCall(len(calls), ev.tool, ev.args or {}, ev.author, ev.elapsed)
            calls.append(c)
            pending.setdefault(ev.tool, []).append(c)
        elif ev.type == "tool_response":
            q = pending.get(ev.tool)
            if q:
                q.pop(0).result = ev.result
    return calls

In [ ]:
%%writefile /kaggle/working/pilot/common/metrics.py
"""Per-task and aggregate metrics shared by all pilots.

Common metric set (identical definitions in every pilot):
  resolved          scorer verdict (Phase 2 pytest + JUnit validation)
  wall_s            TaskResult.duration_seconds (setup + agent + verification)
  agent_s           last event timestamp in the trace (time inside the agent loop)
  tool_calls        tool_call events in the trace (all agents, incl. sub-agents)
  budget_tool_calls harness-counted calls (TaskResult.tool_calls; budget-gated tools only)
  repeat_share      share of tool calls identical (same tool + canonical JSON args) to an
                    earlier call in the same task
  malformed_share   malformed / (tool_calls + unparsed_tool_calls); malformed = unknown tool,
                    missing/unexpected arguments, or argument validation errors; unparsed =
                    turns cut off inside an unclosed <|tool_call> (harness nudge type 1)
  failed_edits      edit_file / write_file calls that returned status=error
  skill_calls       calls to ADK skill tools (list/search/load_skill, load_skill_resource,
                    run_skill_script)
Optional localization metrics (need gold locations, see common.tasks.gold_locations):
  viewed_gold_file, first_gold_view_call, edited_gold_file, edited_gold_function
"""
from __future__ import annotations

import json
import re
import statistics
from typing import Any, Iterable

from .diffs import parse_unified_diff
from .trace import Event, ToolCall, pair_tool_calls

SKILL_TOOLS = {"list_skills", "search_skills", "load_skill", "load_skill_resource", "run_skill_script"}
EDIT_TOOLS = {"edit_file", "write_file"}
_MALFORMED_TYPES = {"TypeError", "ValidationError", "KeyError", "JSONDecodeError", "ArgumentError"}
_MALFORMED_PATTERNS = (
    re.compile(r"Tool '.*' not found", re.S),
    re.compile(r"mandatory input parameters are not present"),
    re.compile(r"unexpected keyword argument"),
    re.compile(r"missing \d+ required (positional|keyword)"),
    re.compile(r"cannot be greater than end_line"),
)
_UNPARSED_NUDGE = "tool call finished closing"


def canonical_call(c: ToolCall) -> str:
    return c.tool + "\x00" + json.dumps(c.args, sort_keys=True, default=str)


def _error_info(c: ToolCall) -> tuple[bool, str, str]:
    r = c.result_json()
    if r is None:
        return False, "", ""
    if r.get("status") == "error":
        return True, str(r.get("error_type", "")), str(r.get("error_message") or r.get("error") or "")
    return False, "", ""


def is_malformed(c: ToolCall) -> bool:
    is_err, etype, msg = _error_info(c)
    if not is_err:
        return False
    if etype in _MALFORMED_TYPES:
        return True
    return any(p.search(msg) for p in _MALFORMED_PATTERNS)


def call_touches_path(c: ToolCall, path: str) -> bool:
    """True if a tool call reads/views `path` (read_file, or a shell command naming it)."""
    if c.tool == "read_file":
        fp = str(c.args.get("filepath", "")).lstrip("/")
        fp = fp[len("workspace/"):] if fp.startswith("workspace/") else fp
        return fp == path or fp.endswith("/" + path)
    if c.tool == "run_command":
        return path in str(c.args.get("command", ""))
    return False


def task_metrics(
    result: dict[str, Any],
    events: list[Event],
    gold: dict[str, Any] | None = None,
    agent_patch_scopes: Iterable[str] | None = None,
) -> dict[str, Any]:
    """Compute the common metric row for one task.

    result: a TaskResult row (task_results.jsonl line or TaskResult.model_dump()).
    events: parsed trace (common.trace.load_trace / parse_trace).
    gold:   optional GoldLocations.to_dict() for localization metrics.
    agent_patch_scopes: optional dotted names of scopes the agent's patch changed.
    The returned row holds numbers/flags only (no patch, no trace text).
    """
    calls = pair_tool_calls(events)
    n = len(calls)
    seen: set[str] = set()
    repeats = 0
    for c in calls:
        k = canonical_call(c)
        if k in seen:
            repeats += 1
        seen.add(k)
    unparsed = sum(1 for e in events if e.type == "continuation_nudge" and _UNPARSED_NUDGE in e.content)
    nudges = sum(1 for e in events if e.type == "continuation_nudge")
    malformed = sum(1 for c in calls if is_malformed(c))
    edit_calls = [c for c in calls if c.tool in EDIT_TOOLS]
    failed_edits = sum(1 for c in edit_calls if _error_info(c)[0])
    by_tool: dict[str, int] = {}
    for c in calls:
        by_tool[c.tool] = by_tool.get(c.tool, 0) + 1
    agent_s = max((e.elapsed for e in events), default=0.0)
    patch = result.get("agent_patch") or ""
    patch_files = [fd.path for fd in parse_unified_diff(patch)] if patch else []

    row: dict[str, Any] = {
        "instance_id": result.get("task_id") or result.get("instance_id"),
        "resolved": bool(result.get("resolved")),
        "error": (result.get("error_message") or "")[:120] or None,
        "wall_s": round(float(result.get("duration_seconds") or 0.0), 1),
        "agent_s": round(agent_s, 1),
        "tool_calls": n,
        "budget_tool_calls": int(result.get("tool_calls") or 0),
        "llm_calls": int(result.get("total_llm_calls") or 0),
        "total_tokens": int(result.get("total_tokens") or 0),
        "repeat_calls": repeats,
        "repeat_share": round(repeats / n, 4) if n else 0.0,
        "malformed_calls": malformed,
        "unparsed_tool_calls": unparsed,
        "malformed_share": round((malformed + unparsed) / (n + unparsed), 4) if (n + unparsed) else 0.0,
        "edit_calls": len(edit_calls),
        "failed_edits": failed_edits,
        "skill_calls": sum(1 for c in calls if c.tool in SKILL_TOOLS),
        "nudges": nudges,
        "submitted": any(c.tool == "submit_patch" for c in calls),
        "patch_nonempty": bool(patch.strip()),
        "patch_files": len(patch_files),
        "calls_by_tool": by_tool,
    }
    if gold is not None:
        gfiles = list(gold.get("src_files") or [])
        first = next((c.index for c in calls if any(call_touches_path(c, g) for g in gfiles)), None)
        row["viewed_gold_file"] = first is not None
        row["first_gold_view_call"] = first
        row["edited_gold_file"] = bool(set(patch_files) & set(gfiles))
        row["edited_only_gold_files"] = bool(patch_files) and set(patch_files) <= set(gfiles)
        if agent_patch_scopes is not None:
            gfun = {f["dotted"] for f in gold.get("functions") or []}
            row["edited_gold_function"] = bool(gfun & set(agent_patch_scopes)) if gfun else None
    return row


def _mean(xs: list[float]) -> float | None:
    return round(statistics.fmean(xs), 3) if xs else None


def _median(xs: list[float]) -> float | None:
    return round(statistics.median(xs), 3) if xs else None


def aggregate(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Aggregate common metrics over a list of task rows (one condition/run)."""
    n = len(rows)
    tot_calls = sum(r["tool_calls"] for r in rows)
    tot_unparsed = sum(r["unparsed_tool_calls"] for r in rows)
    out: dict[str, Any] = {
        "n_tasks": n,
        "resolved": sum(r["resolved"] for r in rows),
        "resolve_rate": round(sum(r["resolved"] for r in rows) / n, 4) if n else None,
        "wall_s_mean": _mean([r["wall_s"] for r in rows]),
        "wall_s_median": _median([r["wall_s"] for r in rows]),
        "agent_s_mean": _mean([r["agent_s"] for r in rows]),
        "tool_calls_mean": _mean([r["tool_calls"] for r in rows]),
        # pooled shares (calls across all tasks) and per-task means
        "repeat_share_pooled": round(sum(r["repeat_calls"] for r in rows) / tot_calls, 4) if tot_calls else 0.0,
        "malformed_share_pooled": round((sum(r["malformed_calls"] for r in rows) + tot_unparsed) / (tot_calls + tot_unparsed), 4)
        if (tot_calls + tot_unparsed) else 0.0,
        "repeat_share_task_mean": _mean([r["repeat_share"] for r in rows]),
        "malformed_share_task_mean": _mean([r["malformed_share"] for r in rows]),
        "failed_edits_total": sum(r["failed_edits"] for r in rows),
        "edit_calls_total": sum(r["edit_calls"] for r in rows),
        "skill_calls_total": sum(r["skill_calls"] for r in rows),
        "submitted_share": round(sum(r["submitted"] for r in rows) / n, 4) if n else None,
        "patch_nonempty_share": round(sum(r["patch_nonempty"] for r in rows) / n, 4) if n else None,
        "errors": sum(1 for r in rows if r.get("error")),
    }
    for k in ("viewed_gold_file", "edited_gold_file", "edited_only_gold_files", "edited_gold_function"):
        vals = [r[k] for r in rows if r.get(k) is not None]
        if vals:
            out[f"{k}_share"] = round(sum(vals) / len(vals), 4)
    fg = [r["first_gold_view_call"] for r in rows if r.get("first_gold_view_call") is not None]
    if fg:
        out["first_gold_view_call_median"] = _median(fg)
    return out


def paired_flips(a: list[dict[str, Any]], b: list[dict[str, Any]]) -> dict[str, Any]:
    """Paired comparison of two runs over the same tasks (b vs a).

    Returns discordant counts and an exact two-sided McNemar (binomial) p-value.
    """
    ra = {r["instance_id"]: r["resolved"] for r in a}
    rb = {r["instance_id"]: r["resolved"] for r in b}
    common = sorted(set(ra) & set(rb))
    gain = sum(1 for t in common if rb[t] and not ra[t])
    loss = sum(1 for t in common if ra[t] and not rb[t])
    m = gain + loss
    if m == 0:
        p = 1.0
    else:
        from math import comb
        k = min(gain, loss)
        p = min(1.0, 2 * sum(comb(m, i) for i in range(k + 1)) / 2 ** m)
    return {"n_common": len(common), "b_only": gain, "a_only": loss, "both": sum(1 for t in common if ra[t] and rb[t]),
            "delta": (sum(rb[t] for t in common) - sum(ra[t] for t in common)), "mcnemar_exact_p": round(p, 4)}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/__init__.py

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/hints.py
"""Oracle-hint rendering for Pilot C (locations and test names only, never patch content).

Conditions (nested for localization, orthogonal for tests):
  O0      no hint
  O-file  gold source file(s)
  O-func  gold file(s) + enclosing function/class names (with def line)
  O-test  scorer-required test names + their test file(s)
  O-all   O-func + O-test

Hints are appended to problem_statement in a LOCAL copy of the tasks used only
by the local evaluation runner. They must never be placed in a submission.
"""
from __future__ import annotations

import copy

CONDITIONS = ("O0", "O-file", "O-func", "O-test", "O-all")
_HEADER = "## Location hints (verified ground truth for this task)"


def _file_lines(g: dict) -> list[str]:
    out = []
    new = set(g.get("new_src_files") or [])
    for f in g.get("src_files") or []:
        out.append(f"- The fix changes `{f}`" + (" (a new file that does not exist yet)." if f in new else "."))
    if not out:
        out.append("- The fix does not change any existing non-test Python source file.")
    return out


def _func_lines(g: dict) -> list[str]:
    out = []
    by_file: dict[str, list[dict]] = {}
    for fn in g.get("functions") or []:
        by_file.setdefault(fn["file"], []).append(fn)
    for f in g.get("src_files") or []:
        for fn in by_file.get(f, []):
            what = "class body of" if fn["kind"] == "class" else "function"
            out.append(f"- In `{f}`: {what} `{fn['dotted']}` (defined at line {fn['def_line']}).")
        if f in (g.get("module_level_files") or []):
            out.append(f"- In `{f}`: module-level code (imports, constants, or new top-level definitions).")
    return out


def _test_lines(g: dict) -> list[str]:
    names = g.get("test_names") or []
    files = g.get("test_files") or []
    new = set(g.get("new_test_files") or [])
    out = []
    for f in files:
        status = "this test file does not exist yet; it will be added when your fix is verified" if f in new \
            else "this test file exists; the listed tests may be new or changed"
        out.append(f"- Verification runs `{f}` ({status}).")
    if names:
        out.append("- Tests that must pass: " + ", ".join(f"`{n}`" for n in names) + ".")
    return out


def render_hint(g: dict, condition: str) -> str:
    if condition not in CONDITIONS:
        raise ValueError(condition)
    if condition == "O0":
        return ""
    lines: list[str] = []
    if condition in ("O-file", "O-func", "O-all"):
        lines += _file_lines(g)
    if condition in ("O-func", "O-all"):
        lines += _func_lines(g)
    if condition in ("O-test", "O-all"):
        lines += _test_lines(g)
    return "\n".join([_HEADER, *lines])


def rewrite_task(task: dict, g: dict, condition: str) -> dict:
    """Return a copy of `task` with the hint appended to problem_statement."""
    t = copy.deepcopy(task)
    hint = render_hint(g, condition)
    if hint:
        t["problem_statement"] = t["problem_statement"].rstrip() + "\n\n" + hint + "\n"
    return t

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/select_tasks.py
"""Select the pilot tasks (instance IDs only are written; safe to commit).

Rule (fixed before any agent run):
  eligible = exactly 1 changed source file that already exists at base_commit,
             1-2 enclosing functions/classes (function-level oracle well defined),
             gold patch <= 40 changed lines, >= 1 scorer-required test name,
             repo in {fastapi, rich, requests},
             and the GOLD patch resolves the task in the sandbox (gold check JSON, if given)
  pilot15  = stratified random sample: fastapi 7, rich 6, requests 2 (seeded)
  reserve  = next 2 per repo from the same shuffled order (replace tasks whose gold
             patch fails the CPU gold check)
  smoke3   = first pilot task of each repo (used for the smoke test and size timing)

usage: python -I pilot_c/select_tasks.py <gold_all.json> <out task_ids.json> [gold_check.json ...]
"""
import json
import random
import sys

SEED = 20261031
QUOTA = {"fastapi/fastapi": 7, "Textualize/rich": 6, "psf/requests": 2}
RESERVE = 2


def eligible(g):
    return (len(g["src_files"]) == 1 and not g["new_src_files"] and 1 <= len(g["functions"]) <= 2
            and g["gold_added"] + g["gold_removed"] <= 40 and len(g["test_names"]) >= 1 and g["repo"] in QUOTA)


def main():
    G = json.load(open(sys.argv[1]))
    checks = {}
    for path in sys.argv[3:]:
        for r in json.load(open(path))["rows"]:
            checks[r["instance_id"]] = bool(r["gold_resolved"])
    if checks:
        G = {i: g for i, g in G.items() if checks.get(i) is True}
    rng = random.Random(SEED)
    pilot, reserve, smoke, pool_sizes = [], [], [], {}
    for repo, k in QUOTA.items():
        pool = sorted(i for i, g in G.items() if g["repo"] == repo and eligible(g))
        pool_sizes[repo] = len(pool)
        rng.shuffle(pool)
        pilot += pool[:k]
        reserve += pool[k:k + RESERVE]
        smoke.append(pool[0])
    out = {"seed": SEED, "gold_check_sources": sys.argv[3:], "n_gold_checked": len(checks), "rule": __doc__.split("Rule (fixed before any agent run):")[1].split("usage:")[0].strip(),
           "eligible_pool_sizes": pool_sizes, "pilot15": pilot, "reserve": reserve, "smoke3": smoke}
    json.dump(out, open(sys.argv[2], "w"), indent=1)
    print(json.dumps({k: out[k] for k in ("eligible_pool_sizes", "smoke3")}), len(pilot), len(reserve))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/kaggle_runner.py
"""Kaggle-side runner for Pilot C (imported by the notebooks; runs only on Kaggle).

Reproduces the official starter's evaluation path (swegemma Evaluator, subprocess
sandbox, vLLM with max_model_len=32768, tool/reasoning parser gemma4, the same
compaction and context-cache settings as the scorer per HARNESS_README §7.2) and
changes only: model alias, per-task budget (eval_config.yaml), task copies with
oracle hints, results dir, concurrency.

Outputs:
  /kaggle/working/private/...         raw runs (patches, traces) - NEVER commit
  /kaggle/working/pilot_c_<tag>.json  numbers only - commit to results/
"""
from __future__ import annotations

import asyncio
import concurrent.futures
import datetime as _dt
import glob
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))

from common.metrics import aggregate, paired_flips, task_metrics  # noqa: E402
from common.tasks import diff_scopes, gold_locations, load_tasks  # noqa: E402
from common.trace import load_trace  # noqa: E402
from pilot_c.hints import rewrite_task  # noqa: E402

WORK = Path(os.environ.get("PILOT_WORK_DIR", "/kaggle/working"))
PRIVATE = WORK / "private"
ALLOWED = ("gemma-4-31b-it-qat-w4a16-ct", "gemma-4-12b-it-qat-w4a16-ct",
           "gemma-4-e4b-it-qat-w4a16-ct", "gemma-4-e2b-it-qat-w4a16-ct")
SHORT = {m: m.split("-")[2] for m in ALLOWED}  # 31b / 12b / e4b / e2b
MODEL_VLLM_EXTRA_ARGS = {
    "gemma-4-12b-it-qat-w4a16-ct": ("--hf-overrides", '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}'),
}
GPU_ONLY_WHEELS = ("vllm", "flashinfer", "bitsandbytes", "xgrammar", "nvidia_", "quack_kernels", "apache_tvm_ffi")


def _first_dir(*cands: str) -> Path:
    for c in cands:
        for p in sorted(glob.glob(c)):
            if Path(p).is_dir():
                return Path(p)
    raise FileNotFoundError(f"none of {cands} found")


def data_dir() -> Path:
    if os.environ.get("PILOT_DATA_DIR"):  # local testing only
        return Path(os.environ["PILOT_DATA_DIR"])
    return _first_dir("/kaggle/input/competitions/gemma-4-developer-agent", "/kaggle/input/gemma-4-developer-agent")


def wheelhouse_dir() -> Path:
    return _first_dir("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse",
                      "/kaggle/input/gemma-4-developer-agent-wheelhouse")


def set_env() -> None:
    """Same environment variables as the official starter notebook."""
    env = {
        "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1", "TRANSFORMERS_OFFLINE": "1",
        "VLLM_NO_USAGE_STATS": "1", "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
        "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1", "NCCL_DEBUG": "WARN", "LITELLM_LOCAL_MODEL_COST_MAP": "True",
        "LITELLM_TELEMETRY": "False", "LITELLM_LOG": "WARNING", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "OTEL_SDK_DISABLED": "true", "OTEL_PYTHON_DISABLED": "true",
    }
    os.environ.update(env)
    os.environ.setdefault("KAGGLE_SANDBOX_DIR", str(data_dir() / "sandbox"))
    os.environ.setdefault("OPENAI_API_KEY", "EMPTY")
    os.environ.setdefault("LOCAL_API_KEY", "EMPTY")


def install_wheelhouse(gpu: bool = True) -> None:
    """Starter-identical wheel install (cpu mode skips GPU-only wheels)."""
    import importlib
    for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth", "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for pth in glob.glob(pat):
            try:
                os.unlink(pth)
            except OSError:
                pass
    tmp = Path("/tmp/wheelhouse")
    tmp.mkdir(parents=True, exist_ok=True)
    for w in wheelhouse_dir().glob("*.whl"):
        if "cutlass" in w.name.lower() or (not gpu and w.name.startswith(GPU_ONLY_WHEELS)):
            continue
        name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
        t = tmp / name
        if not t.exists():
            try:
                os.symlink(w, t)
            except OSError:
                shutil.copyfile(w, t)
    wheels = sorted(str(w) for w in tmp.glob("*.whl"))
    print(f"installing {len(wheels)} wheels")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels], check=True)
    importlib.invalidate_caches()


def quiet_logs() -> None:
    import logging
    for name in ("LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm", "httpx", "httpcore", "openai", "urllib3",
                 "opentelemetry", "opentelemetry.context"):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if "litellm" in name.lower():
            lg.propagate = False
            lg.handlers.clear()


def run_sync(fn, *args, **kwargs):
    """Run an async callable from a notebook (same helper as the starter)."""
    coro_fn = lambda: fn(*args, **kwargs)  # noqa: E731
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())


def env_info() -> dict[str, Any]:
    info: dict[str, Any] = {"python": platform.python_version(), "nproc": os.cpu_count()}
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        info["gpus"] = out
    except Exception:  # noqa: BLE001
        info["gpus"] = []
    try:
        import importlib.metadata as md
        info["versions"] = {p: md.version(p) for p in ("swegemma", "adk-eval-core", "adk-submission", "google-adk", "vllm")
                            if _has_dist(md, p)}
    except Exception:  # noqa: BLE001
        pass
    return info


def _has_dist(md, p: str) -> bool:
    try:
        md.version(p)
        return True
    except Exception:  # noqa: BLE001
        return False


# --------------------------------------------------------------------------- tasks & hints

def build_gold(task_ids: list[str]) -> dict[str, dict]:
    d = data_dir()
    tasks = {t["instance_id"]: t for t in load_tasks(d / "tasks.jsonl")}
    out = {}
    for iid in task_ids:
        out[iid] = gold_locations(tasks[iid], d / "snapshots" / f"{iid}.tgz").to_dict()
    return out


def write_task_copy(task_ids: list[str], gold: dict[str, dict], condition: str) -> Path:
    """Local hinted copy of tasks.jsonl for one condition (kept under PRIVATE)."""
    d = data_dir()
    tasks = {t["instance_id"]: t for t in load_tasks(d / "tasks.jsonl")}
    PRIVATE.mkdir(parents=True, exist_ok=True)
    path = PRIVATE / f"tasks_{condition}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for iid in task_ids:
            f.write(json.dumps(rewrite_task(tasks[iid], gold[iid], condition)) + "\n")
    return path


# --------------------------------------------------------------------------- agent & server

def prepare_agent_dir(model: str, budget: dict[str, Any]) -> Path:
    """Official sample_submission with the model alias swapped and the sample's dummy LoRA
    adapters removed for EVERY size (starter logic for non-31B; for 31B too, so all sizes run the
    plain base model), and eval_config.yaml replaced by the pilot budget."""
    assert model in ALLOWED, model
    src = data_dir() / "sample_submission"
    dst = WORK / f"agent_{SHORT[model]}"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    shutil.rmtree(dst / "adapters", ignore_errors=True)
    for y in [dst / "agent.yaml", *sorted((dst / "sub_agents").glob("*.yaml"))]:
        if y.is_file():
            txt = y.read_text(encoding="utf-8")
            txt = re.sub(r"^(\s*model:\s*).*$", rf"\g<1>{model}", txt, flags=re.M)
            txt = re.sub(r"^\s*adapter:\s*.*\n?", "", txt, flags=re.M)
            y.write_text(txt, encoding="utf-8")
    lines = ["evaluation:"] + [f"  {k}: {v}" for k, v in budget.items()]
    (dst / "eval_config.yaml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return dst


def describe_agent(agent_dir: Path) -> dict[str, Any]:
    import yaml
    a = yaml.safe_load((agent_dir / "agent.yaml").read_text().replace("!include", "#include")) or {}
    samp = yaml.safe_load((agent_dir / "configs" / "sampling.yaml").read_text())
    tools = [t if isinstance(t, str) else "agent_tool:" + str(t.get("agent_tool", {}).get("config_path")) for t in a.get("tools", [])]
    return {"model": a.get("model"), "tools": tools, "sampling": samp,
            "eval_config": yaml.safe_load((agent_dir / "eval_config.yaml").read_text())}


def start_server(agent_dir: Path):
    """Starter-identical vLLM launch. Returns (server, models, startup_seconds)."""
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from adk_submission.discovery import discover_declared_models
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

    litellm.drop_params = True
    declared = discover_declared_models(agent_dir)
    assert len(declared) == 1, declared
    model = next(iter(declared))
    assert model in ALLOWED, model
    model_path = Path(f"/kaggle/input/models/google/gemma-4/other/{model}/2")
    if not model_path.exists():
        model_path = _first_dir(f"/kaggle/input/models/google/gemma-4/*/{model}/*", f"/kaggle/input/gemma-4/*/{model}/*")
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    amap = getattr(adapters, "adapters", None) or {}
    ranks = [r for info in amap.values() if (r := getattr(info, "rank", None)) is not None and r > 0]
    n = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp = 4 if n >= 4 else (2 if n >= 2 else 1)
    cfg = VllmConfig(
        model=str(model_path), port=8000, host="127.0.0.1", tool_call_parser="gemma4", reasoning_parser="gemma4",
        max_model_len=32768, dtype="bfloat16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "auto",
        gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=bool(amap), max_loras=max(1, len(amap)),
        max_lora_rank=max(ranks) if ranks else 128, scheduler_reserve_full_isl=False, tensor_parallel_size=tp,
        startup_timeout=60 * 20, extra_args=MODEL_VLLM_EXTRA_ARGS.get(model, ()),
    )
    t0 = time.perf_counter()
    server = VllmServer(cfg, adapter_manifest=adapters)
    server.start()
    startup = time.perf_counter() - t0
    models = server.create_model_registry(aliases=[model], model_prefix="openai/", api_key=os.environ.get("LOCAL_API_KEY", "EMPTY"))
    print(f"vLLM up for {model} in {startup:.0f}s (tp={tp})")
    return server, models, adapters, startup


# --------------------------------------------------------------------------- evaluation

def make_eval_config(agent_dir: Path, models, adapters, tasks_path: Path, task_ids: list[str], results_dir: Path,
                     concurrency: int, sandbox: str = "subprocess"):
    import yaml
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import EvalConfig, build_submission_limits

    raw = yaml.safe_load((agent_dir / "eval_config.yaml").read_text()) or {}
    ev = raw.get("evaluation", raw)
    turns = ev.get("max_turns", ev.get("max_llm_calls"))
    limits, gen = build_submission_limits()
    d = data_dir()
    return EvalConfig(
        tasks_path=tasks_path, snapshots_dir=d / "snapshots", results_dir=results_dir, submission_dir=agent_dir,
        models=models, sandbox=sandbox,
        timeout_seconds=int(ev.get("timeout_seconds", 300)),
        verification_timeout_seconds=int(ev.get("verification_timeout_seconds", 300)),
        max_time_minutes=float(ev.get("max_time_minutes", 60.0)), max_tool_calls=int(ev.get("max_tool_calls", 100)),
        max_turns=int(turns) if turns is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336,
                                                        event_retention_size=5),
        graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), wheels_dir=d / "wheels",
        verbose=False, task_ids=list(task_ids), concurrency=concurrency, display_mode="quiet",
    )


def _result_dict(r) -> dict[str, Any]:
    try:
        d = r.model_dump(exclude={"trace"})
    except Exception:  # noqa: BLE001
        d = dict(r.__dict__)
        d.pop("trace", None)
    d.setdefault("instance_id", getattr(r, "instance_id", None) or d.get("task_id"))
    return d


def collect_rows(results, results_dir: Path, gold: dict[str, dict]) -> list[dict[str, Any]]:
    d = data_dir()
    rows = []
    for r in results:
        rd = _result_dict(r)
        iid = rd.get("instance_id") or rd.get("task_id")
        tpath = rd.get("trace_path") or str(results_dir / "traces" / f"trace_{iid}.json")
        events = load_trace(tpath) if tpath and Path(tpath).exists() else []
        patch = rd.get("agent_patch") or ""
        scopes = diff_scopes(patch, d / "snapshots" / f"{iid}.tgz") if patch.strip() else []
        row = task_metrics(rd, events, gold=gold.get(iid), agent_patch_scopes=scopes)
        row["trace_found"] = bool(events)
        rows.append(row)
    return rows


def run_condition(agent_dir: Path, models, adapters, task_ids: list[str], gold: dict[str, dict], condition: str,
                  replicate: int, concurrency: int) -> dict[str, Any]:
    from swegemma.evaluate import Evaluator
    model_short = agent_dir.name.replace("agent_", "")
    tasks_path = write_task_copy(task_ids, gold, condition)
    rdir = PRIVATE / "runs" / model_short / f"{condition}_r{replicate}"
    if rdir.exists():
        shutil.rmtree(rdir)
    rdir.mkdir(parents=True)
    cfg = make_eval_config(agent_dir, models, adapters, tasks_path, task_ids, rdir, concurrency)
    t0 = time.perf_counter()
    res = run_sync(Evaluator(cfg).run)
    batch = time.perf_counter() - t0
    rows = collect_rows(res.task_results, rdir, gold)
    agg = aggregate(rows)
    print(f"[{model_short} {condition} r{replicate}] resolved {agg['resolved']}/{agg['n_tasks']} "
          f"batch {batch/60:.1f} min, mean wall {agg['wall_s_mean']}s, calls {agg['tool_calls_mean']}")
    return {"model": model_short, "condition": condition, "replicate": replicate, "concurrency": concurrency,
            "batch_wall_s": round(batch, 1), "aggregate": agg, "rows": rows}


def save_summary(path: Path, summary: dict[str, Any]) -> None:
    runs = summary.get("runs", [])
    comps = []
    base = {(r["model"]): r for r in runs if r["condition"] == "O0" and r["replicate"] == 1}
    for r in runs:
        b = base.get(r["model"])
        if b is not None and r is not b:
            comps.append({"model": r["model"], "a": "O0_r1", "b": f"{r['condition']}_r{r['replicate']}",
                          **paired_flips(b["rows"], r["rows"])})
    summary["paired_vs_O0_r1"] = comps
    summary["saved_utc"] = _dt.datetime.utcnow().isoformat(timespec="seconds")
    path.write_text(json.dumps(summary, indent=1))
    print(f"saved {path}")


# --------------------------------------------------------------------------- CPU gold check

def gold_check(task_ids: list[str], out_path: Path, timeout_s: int = 300) -> dict[str, Any]:
    """CPU-only: apply each task's GOLD patch in the scorer's Phase-2 sandbox and record resolved.
    Tells us which tasks are solvable at all in the Kaggle subprocess sandbox. Numbers only."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.deduplication import resolve_task_snapshot_paths
    from swegemma.evaluate import Evaluator
    from swegemma.harness.verification import verify_task
    from swegemma.models import load_tasks as sg_load

    d = data_dir()
    PRIVATE.mkdir(parents=True, exist_ok=True)
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=PRIVATE / "gold_check",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     timeout_seconds=timeout_s, verification_timeout_seconds=timeout_s, wheels_dir=d / "wheels",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    ev = Evaluator(cfg)
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    rows = []
    for i, iid in enumerate(task_ids, 1):
        t = tasks[iid]
        snap, base, patch = resolve_task_snapshot_paths(d / "snapshots", iid, t.repo)
        t0 = time.perf_counter()
        try:
            r = run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                         agent_patch=t.patch, agent_error=None, trace=None, start_time=t0)
            ok, err = bool(r.resolved), (r.error or "")[:160] or None
        except Exception as e:  # noqa: BLE001
            ok, err = False, f"{type(e).__name__}: {e}"[:160]
        dt = time.perf_counter() - t0
        rows.append({"instance_id": iid, "gold_resolved": ok, "verify_s": round(dt, 1), "error": err})
        print(f"[{i}/{len(task_ids)}] {iid} gold_resolved={ok} {dt:.0f}s {err or ''}")
        out_path.write_text(json.dumps({"rows": rows, "env": env_info()}, indent=1))
    return {"rows": rows}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/task_ids.json
{
 "status": "PROVISIONAL: pilot15 must be re-drawn with the Kaggle gold check (notebook 00) before the GPU pilot",
 "seed": 20261031,
 "rule": "eligible = exactly 1 changed source file that already exists at base_commit,\n             1-2 enclosing functions/classes (function-level oracle well defined),\n             gold patch <= 40 changed lines, >= 1 scorer-required test name,\n             repo in {fastapi, rich, requests},\n             and the GOLD patch resolves the task in the sandbox (gold check JSON, if given)\n  pilot15  = stratified random sample: fastapi 7, rich 6, requests 2 (seeded)\n  reserve  = next 2 per repo from the same shuffled order (replace tasks whose gold\n             patch fails the CPU gold check)\n  smoke3   = first pilot task of each repo (used for the smoke test and size timing)",
 "selection_without_gold_check": {
  "eligible_pool_sizes": {
   "fastapi/fastapi": 33,
   "Textualize/rich": 27,
   "psf/requests": 10
  },
  "pilot15": [
   "fastapi_15588",
   "fastapi_13537",
   "fastapi_14301",
   "fastapi_14360",
   "fastapi_14303",
   "fastapi_15763",
   "fastapi_11194",
   "rich_3454",
   "rich_3006",
   "rich_4079",
   "rich_4076",
   "rich_3894",
   "rich_4075",
   "requests_7502",
   "requests_7328"
  ],
  "reserve": [
   "fastapi_14356",
   "fastapi_9753",
   "rich_3470",
   "rich_3718",
   "requests_7309",
   "requests_6644"
  ]
 },
 "local_gold_check_note": "local subprocess sandbox (python 3.13); only tasks already checked locally were eligible",
 "pilot15": [
  "fastapi_9753",
  "fastapi_13537",
  "fastapi_11194",
  "fastapi_14301",
  "rich_3470",
  "rich_3718",
  "rich_3894",
  "rich_4076",
  "rich_4075",
  "rich_3006",
  "requests_7427",
  "requests_7309"
 ],
 "reserve": [
  "requests_7315"
 ],
 "eligible_pool_sizes_local_checked": {
  "fastapi/fastapi": 4,
  "Textualize/rich": 6,
  "psf/requests": 3
 },
 "smoke3_gpu": [
  "fastapi_15588",
  "rich_3454",
  "requests_7309"
 ]
}

In [ ]:
import os, sys, time, json
from pathlib import Path
sys.path.insert(0, '/kaggle/working/pilot')
SESSION_T0 = time.time()
from pilot_c import kaggle_runner as kr
kr.set_env()
kr.install_wheelhouse(gpu=True)
kr.quiet_logs()
IDS = json.loads(Path('/kaggle/working/pilot/pilot_c/task_ids.json').read_text())
print(kr.env_info())

In [ ]:
# ---- configuration (only edit here)
TASKS = IDS['pilot15']
BUDGET = dict(timeout_seconds=120, max_tool_calls=50, max_time_minutes=15)
CONCURRENCY = 5
MAX_SESSION_HOURS = 3.9                        # hard ceiling agreed for the pilot is 4 L4x4 hours
RUN_E2B = False
conds = [('O0', 1), ('O-func', 1), ('O-all', 1), ('O0', 2)]
PLAN = [('gemma-4-e4b-it-qat-w4a16-ct', conds)] + ([('gemma-4-e2b-it-qat-w4a16-ct', conds)] if RUN_E2B else [])

In [ ]:
# ---- the experiment loop: one vLLM server per model, conditions run as separate Evaluator batches
summary = {"pilot": "C", "notebook": "pilot_e4b", "env": kr.env_info(), "budget": BUDGET, "task_ids": TASKS,
           "plan": PLAN, "concurrency": CONCURRENCY, "servers": [], "runs": [], "skipped": []}
out = Path('/kaggle/working/pilot_c_pilot_e4b.json')
gold = kr.build_gold(TASKS)
for model, conds in PLAN:
    if (time.time() - SESSION_T0) / 3600 > MAX_SESSION_HOURS:
        summary["skipped"].append({"model": model, "reason": "session time guard"}); continue
    agent_dir = kr.prepare_agent_dir(model, BUDGET)
    summary["agent"] = kr.describe_agent(agent_dir)
    server, models, adapters, startup = kr.start_server(agent_dir)
    summary["servers"].append({"model": model, "startup_s": round(startup, 1)})
    try:
        for cond, rep in conds:
            if (time.time() - SESSION_T0) / 3600 > MAX_SESSION_HOURS:
                summary["skipped"].append({"model": model, "condition": cond, "replicate": rep, "reason": "session time guard"}); continue
            run = kr.run_condition(agent_dir, models, adapters, TASKS, gold, cond, rep, CONCURRENCY)
            summary["runs"].append(run)
            summary["session_elapsed_h"] = round((time.time() - SESSION_T0) / 3600, 3)
            kr.save_summary(out, summary)
    finally:
        server.stop()
summary["session_elapsed_h"] = round((time.time() - SESSION_T0) / 3600, 3)
kr.save_summary(out, summary)
print('L4x4 hours used by this session (wall clock):', summary["session_elapsed_h"])

**When it finishes:** download `/kaggle/working/pilot_c_pilot_e4b.json` (numbers only: per-task flags/counts, no patches or traces)
and commit it to `results/` on branch `pilot/c-bottleneck`. Everything under `/kaggle/working/private/` (hinted task copies,
patches, traces) is competition-derived: keep it on Kaggle, never commit it.